# LLMの文章に「見えない透かし」を入れて、自分で検出する実験
緑リスト透かし（Kirchenbauer et al., ICML 2023）を、transformers標準機能で動かします。

**準備**: メニュー「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU を選択（CPUでも動きますが遅め）。
上から順にセルを実行してください。初回はモデルのダウンロード（約6GB）があります。

In [ ]:
%pip install -q -U transformers accelerate

In [ ]:
# セットアップ: モデル読み込みと透かしの設定
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          WatermarkingConfig, WatermarkDetector,
                          WatermarkLogitsProcessor)
from transformers.generation import LogitsProcessorList
print("transformers:", __import__("transformers").__version__)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # T4 GPUで快適に動く上限クラス。軽くしたい場合は 1.5B に
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 透かしの設定 = 秘密鍵一式（検出側と完全一致している必要がある）
WM = WatermarkingConfig(
    greenlist_ratio=0.25,   # 語彙の25%を「緑」に
    bias=2.5,               # 緑トークンへの加点
    hashing_key=20260811,   # 秘密鍵（ただの整数。これが実印）
    seeding_scheme="lefthash",
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
model.eval()
detector = WatermarkDetector(model_config=model.config, device=DEVICE, watermarking_config=WM)  # 生成と同じデバイス必須

# 透かしの注入は「明示的なプロセッサ渡し」方式(バージョン差の影響を受けない)
WM_PROC = WatermarkLogitsProcessor(
    vocab_size=model.config.vocab_size, device=DEVICE,
    greenlist_ratio=0.25, bias=2.5, hashing_key=20260811, seeding_scheme="lefthash")

def _s(x):
    if hasattr(x, "item"): return x.item()
    try: return x[0]
    except (TypeError, IndexError): return x

def show(label, ids):
    r = detector(ids.to(DEVICE), return_dict=True)
    gf, z = float(_s(r.green_fraction)), float(_s(r.z_score))
    pred, n = bool(_s(r.prediction)), int(_s(r.num_tokens_scored))
    print(f"{label:<26} 緑率={gf:5.1%}  z={z:6.2f} → {'透かし検出' if pred else '検出せず'} (評価トークン数 {n})")

def generate(prompt, watermark, max_new_tokens=450, seed=0):
    msgs = [{"role": "user", "content": prompt}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    torch.manual_seed(seed)
    kw = dict(max_new_tokens=max_new_tokens, do_sample=True, temperature=0.8,
              top_p=0.95, pad_token_id=tok.eos_token_id)
    if watermark: kw["logits_processor"] = LogitsProcessorList([WM_PROC])
    out = model.generate(**enc, **kw)
    new = out[:, enc["input_ids"].shape[1]:]
    return tok.decode(new[0], skip_special_tokens=True), new.to("cpu")

def detect_text(text):
    ids = tok(text, return_tensors="pt").input_ids
    if ids.shape[1] < 20:
        print("短すぎます（20トークン以上必要）。短文で検出できないこと自体が透かしの限界のひとつです。")
        return
    show(f"貼り付けテキスト({ids.shape[1]}tk)", ids)

print("準備OK:", MODEL_ID, "/", DEVICE)

## 実験1: 同じお題を「透かしなし」「透かしあり」で生成して読み比べ→検出
2つの文章は読んでも区別できないはずです。検出器だけが白黒を付けられます。

In [ ]:
PROMPT = "秋の京都をひとり旅する魅力について、350字程度のエッセイを日本語で書いてください。"

text_plain, ids_plain = generate(PROMPT, watermark=False)
text_wm, ids_wm = generate(PROMPT, watermark=True)

print("--- 透かしなし ---\n" + text_plain)
print()
print("--- 透かしあり ---\n" + text_wm)
print()
show("透かしなし", ids_plain)
show("透かしあり", ids_wm)

## 実験2: 手で編集すると透かしはどこまで残るか
下のセルの `EDITED_TEXT` に、実験1の「透かしあり」の文章を貼り付けて、
少しずつ手で直しながら（語尾を変える・文を入れ替える・書き足す）何度か実行してください。
z値がじわじわ下がっていきます。別のAIに「同じ内容で言い換えて」させた文章を貼ると、ほぼ一発で消えます。

In [ ]:
EDITED_TEXT = """
（ここに透かしありの文章を貼り付けて、手で編集してから実行）
"""

detect_text(EDITED_TEXT)

## 実験3: コードには埋め込めるのか（低エントロピー実験）
FizzBuzzをあえて透かしオンで書かせます。続きがほぼ一意に決まるコードでは、
抽選に癖をつける余地がなく、文章のように緑率が上がらないはずです。

In [ ]:
CODE_PROMPT = "1から100までのFizzBuzzを出力するPython関数を書いてください。説明は不要、コードのみ。"
code_text, code_ids = generate(CODE_PROMPT, watermark=True, max_new_tokens=250)
print(code_text)
print()
show("コード(透かしオン)", code_ids)

## 実験4: 鍵が違えば、同じ文章でも見えない
検出器の鍵を1つ変えて、実験1の「透かしあり」文章をもう一度判定します。
ベンダーごとに透かしが別物＝「万能AI検出器」がこの方式から生まれない理由の実演です。

In [ ]:
WM2 = WatermarkingConfig(greenlist_ratio=0.25, bias=2.5,
                         hashing_key=99999999, seeding_scheme="lefthash")  # 鍵だけ変更
detector2 = WatermarkDetector(model_config=model.config, device=DEVICE, watermarking_config=WM2)
r = detector2(ids_wm.to(DEVICE), return_dict=True)
gf, z = float(_s(r.green_fraction)), float(_s(r.z_score))
print(f"別の鍵で判定:              緑率={gf:5.1%}  z={z:6.2f} → {'透かし検出' if bool(_s(r.prediction)) else '検出せず'}")
show("正しい鍵で判定(参考)", ids_wm)

---
記事: タビの足袋 https://taviko.com/ ／ 方式: Kirchenbauer et al., "A Watermark for Large Language Models" (ICML 2023) https://arxiv.org/abs/2301.10226